In [ ]:
import ROOT, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

PSD_X1=-284.5
PSD_Y1=-310.7
PSD_X2=-298.5
PSD_Y2=-324.7

In [ ]:
dirDir= "/home/users/l/lebrun/"
dataDir = "/dpnc/beegfs/dampe/tryf/MC_Skimmed/MC_gamma/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
filename1 = dataDir + "DmlNtup_gamma_part*_TQ_FullSel.root"
filename2 = dataDir + "DmlNtup_gamma_100GeV_1TeV_part*_TQ_FullSel.root"
filename3 = dataDir + "DmlNtup_gamma_1TeV_10TeV_part*_MPhotonSelProj_TQ_FullSel.root"

filenames_LET = [filename1, filename2]
filenames_HET = [filename1, filename2, filename3]

columns = ["tt_ekin", "tt_bgoTotalEcorr_GeV", "theta_True", "tt_isHET", "tt_isLET", 
           "phi_True", "theta_BestSTK", "phi_BestSTK"]

In [ ]:
def load_and_filter(filenames=None, LET=False, HET=False, columns=[]):
    """ Loads root files and applies the selection algorithm on gamma rays
    
    Args:
        files (str or list): Path to the root files or a list of root files.
        LET (bool): If True OR if both LET and HET are false, includes the LET events. 
        HET (bool): If True OR if both LET and HET are false, includes the HET events.  
        
    Returns:
        pd.DataFrame: A DataFrame containing the filtered data."""

    # Default behaviour if no filenames are provided
    if not filenames :
        # If HET selected, include all files.
        # If only LET selected, include only LET files.
        # If both LET and HET are false, include all files by default.
        if HET :
            filenames = filenames_HET
        else :
            filenames = filenames_LET

    if isinstance(filenames, str):
        files = sorted(glob.glob(filenames))
    elif isinstance(filenames, list):
        files = []
        for filename in filenames:
            files += sorted(glob.glob(filename))

    print(files)

    # Pass the list of files to RDataFrame
    rdf = ROOT.RDataFrame("DmlNtup", files)
    print("created RDataFrame")
    print(rdf.GetColumnNames())

    if (not LET) and (not HET):
        # If both LET and HET are false, include all events
        LET = True
        HET = True

    if LET and not HET:
        # Apply the selection for LET events
        rdf = rdf.Filter("tt_isLET == 1")
    elif HET and not LET:
        # Apply the selection for HET events
        rdf = rdf.Filter("tt_isHET == 1")

    print("filtered LET/HET")

    # Filter on BGO trajectory
    rdf = rdf.Filter(f"fabs(tt_bgoRecSlopeX * {PSD_X2} + tt_bgoRecInterceptX) < 420").Filter(
        f"fabs(tt_bgoRecSlopeY * {PSD_Y2} + tt_bgoRecInterceptY) < 420")

    print("filtered BGO trajectory")

    # Filter on Ecore
    rdf = rdf.Filter("tt_Ecore3 <= 1").Filter("tt_Ecore3 > 0.9")

    print("filtered Ecore")

    # With tracks
    rdf = rdf.Filter("nnTracks > 0")

    print("filtered tracks")

    # Geometry cuts
    rdf = rdf.Filter("tt_F0 <0.35").Filter("tt_F1<0.45")
    rdf = rdf.Filter("CUT_TRACK_NO_HOLES_Best == 1").Filter("theta_BestSTK < 60").Filter("tt_Xtrl < 100")
    rdf = rdf.Filter("STKPlaneCut_Best == 1").Filter("fabs(X_ProjectionBest_X1) < 392").Filter(
        "fabs(X_ProjectionBest_X2) < 392").Filter("fabs(Y_ProjectionBest_Y1) < 392").Filter(
        "fabs(Y_ProjectionBest_Y2) < 392")
    
    print("filtered geometry cuts")
    
    # Cuts on charge
    rdf = rdf.Filter("(ProjSTKPSD_countX_Best<=1 && ProjSTKPSD_countY_Best<=1) || (ProjSTKPSD_count_Best/log(tt_bgoTotalE_GeV)<=0.65)")

    print("filtered charge cuts")

    # Cut on CNN
    rdf = rdf.Filter("CNN_linear>1.84")

    print("filtered CNN")

    # Sanity cuts TODO : add Y_ProjectionBest_Y1_E_3bar etc cut 
    rdf = rdf.Filter("tt_maxEX1Psd/tt_bgoTotalE_GeV<1.5").Filter("tt_maxEX2Psd/tt_bgoTotalE_GeV<1.5").Filter("tt_maxEY1Psd/tt_bgoTotalE_GeV<1.5").Filter("tt_maxEY2Psd/tt_bgoTotalE_GeV<1.5")
    rdf = rdf.Filter("tt_QcloClusterXTrackCorr/tt_bgoTotalE_GeV<0.5").Filter("tt_QcloClusterYTrackCorr/tt_bgoTotalE_GeV<0.5").Filter("(X_ProjectionBest_X1_E+X_ProjectionBest_X2_E)/tt_bgoTotalE_GeV<1.").Filter(
        "(Y_ProjectionBest_Y1_E+Y_ProjectionBest_Y2_E)/tt_bgoTotalE_GeV<1.").Filter("(X_ProjectionBest_X1_E_1bar+X_ProjectionBest_X2_E_1bar)/tt_bgoTotalE_GeV<1.").Filter("(Y_ProjectionBest_Y1_E_1bar+Y_ProjectionBest_Y2_E_1bar)/tt_bgoTotalE_GeV<1.").Filter(
            "(X_ProjectionBest_X1_E_3bar+X_ProjectionBest_X2_E_3bar)/tt_bgoTotalE_GeV<1.").Filter("(Y_ProjectionBestY1_E_3bar+Y_ProjectionBestY2_E_3bar)/tt_bgoTotalE_GeV<1.").Filter("Etrack_STK_Lay0_02mm_Best<exp(2.8*tt_bgoTotalE_GeV)").Filter("Etrack_STK_Lay1_02mm_Best<exp(2.8*tt_bgoTotalE_GeV)").Filter(
                "Etrack_STK_Lay0_1mm_Best<exp(2.8*tt_bgoTotalE_GeV)").Filter("Etrack_STK_Lay1_1mm_Best<exp(2.8*tt_bgoTotalE_GeV)").Filter("Etrack_STK_Lay0_5mm_Best<exp(3.8*tt_bgoTotalE_GeV)").Filter("Etrack_STK_Lay1_5mm_Best<exp(3.8*tt_bgoTotalE_GeV)").Filter("Etrack_STK_Lay0_10mm_Best<exp(4.0*tt_bgoTotalE_GeV)").Filter(
                    "Etrack_STK_Lay1_10mm_Best<exp(4.0*tt_bgoTotalE_GeV)")
    print("filtered sanity cuts")
    # Cut on BDT : correct ? TODO : assess FilterAvailable
    rdf = rdf.Filter("(tt_bgoTotalE_GeV<=100 && BDT_cleaned>0.18-0.038*log(tt_bgoTotalE_GeV) ) || ((tt_bgoTotalE_GeV>100) && (tt_bgoTotalE_GeV<=1000) && (BDT_cleaned>0.05-0.035*log(0.01*tt_bgoTotalE_GeV)) ) || (tt_bgoTotalE_GeV>1000 and BDT_cleaned>0.05-0.02*log(0.001*tt_bgoTotalE_GeV)) ")
    #rdf = rdf.Filter("(tt_bgoTotalE_GeV<=100 && BDT_new>0.18-0.038*log(tt_bgoTotalE_GeV) ) || ((tt_bgoTotalE_GeV>100) && (tt_bgoTotalE_GeV<=1000) && (BDT_new>0.05-0.035*log(0.01*tt_bgoTotalE_GeV)) ) || (tt_bgoTotalE_GeV>1000 and BDT_new>0.05-0.02*log(0.001*tt_bgoTotalE_GeV)) ")

    print("filtered BDT")

    print(f"Number of events after selection: {rdf.Count().GetValue()}")

    # Convert to DataFrame
    df = rdf.AsNumpy(columns)
    print("converted to numpy")
    df = pd.DataFrame(df)
    print("converted to pandas")
    
    return df

In [ ]:
df = load_and_filter(filename3, LET=False, HET=True, columns=columns)

In [ ]:
df.to_csv(resultPath + "MC_let_filtered.csv", index=False)